In [ ]:
import tensorflow as tf
import pandas as pd
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

In [ ]:
!wget -q https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget -q https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_data = pd.read_csv(
    "train-data.tsv",
    sep="\t",
    names=["label", "text"]
)

test_data = pd.read_csv(
    "valid-data.tsv",
    sep="\t",
    names=["label", "text"]
)

print(train_data.head())
print(test_data.head())
print(train_data.shape)
print(test_data.shape)

In [ ]:
print(train_data.head())
print(test_data.head())

print(train_data.shape)
print(test_data.shape)

In [ ]:
train_texts = train_data["text"].astype(str).values
test_texts = test_data["text"].astype(str).values

train_labels = train_data["label"].map({
    "ham": 0,
    "spam": 1
}).values

test_labels = test_data["label"].map({
    "ham": 0,
    "spam": 1
}).values

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(
    num_words=10000,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(train_texts)

train_sequences = tokenizer.texts_to_sequences(train_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)

max_length = 100

train_padded = pad_sequences(
    train_sequences,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

test_padded = pad_sequences(
    test_sequences,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

In [ ]:
model = keras.Sequential([
    keras.layers.Embedding(
        input_dim=10000,
        output_dim=64,
        input_length=max_length
    ),

    keras.layers.Bidirectional(
        keras.layers.LSTM(64)
    ),

    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dropout(0.4),

    keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
history = model.fit(
    train_padded,
    train_labels,
    epochs=8,
    batch_size=32,
    validation_data=(test_padded, test_labels),
    verbose=1
)

In [ ]:
def predict_message(message):
    sequence = tokenizer.texts_to_sequences([message])

    padded = pad_sequences(
        sequence,
        maxlen=max_length,
        padding="post",
        truncating="post"
    )

    probability = float(model.predict(padded, verbose=0)[0][0])

    if probability >= 0.5:
        return [probability, "spam"]
    else:
        return [probability, "ham"]

In [ ]:
messages = [
    "how are you doing today",
    "sale today! to stop texts call 98912460324",
    "i dont want to go. can we try it a different day? available sat",
    "our new mobile video service is live. just install on your phone to start watching.",
    "you have won £1000 cash! call to claim your prize.",
    "i'll bring it tomorrow. don't forget the milk.",
    "wow, is your arm alright. that happened to me one time too"
]

for message in messages:
    print(message)
    print(predict_message(message))
    print()

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
